# Special Cases

This notebook demonstrates edge cases that rarely occur in a normal election run but trigger specific simulation behaviour.

**Principle:** 1 constituency, 100 votes → vote count = percentage. Winning threshold: **52 %** (default: `ballot_majority_margin = 2 %`).

In [ ]:
import pandas as pd

from ipres import (
    Election, ElectionConfig,
    Ballot, DrawOfLots, DrawLotsStrategy,
    ElectionEvaluator,
)
from ipres.constituencies_config import ConstituenciesConfig

# 1 constituency, 100 votes → vote count = percentage
cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['C'],
    'constituency_size': [100],
    'turnout_percent':   [100.0],
}))

---
## Case 1: Only One Party — `ValueError`

`Ballot.run()` requires at least two contestants. Starting an election with only one party raises a `ValueError`.

In [ ]:
config_1 = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A'],
)

try:
    election_1 = Election(electionConfig=config_1)
    election_1.start(votes={'A': 100})
except ValueError as e:
    print(f"ValueError: {e}")

---
## Case 2a: Two Parties — Win in the First Round

| Party | Votes | Share |
|-------|:-----:|:-----:|
| A     | 53    | 53 %  |
| B     | 47    | 47 %  |

A exceeds the threshold of **52 %** directly → wins in round 1.

In [ ]:
config_2a = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
)

election_2a = Election(electionConfig=config_2a)
r1_2a = election_2a.start(votes={'A': 53, 'B': 47})

print(f"Threshold: {config_2a.getBallotMajorityPercent():.0f} %")
print(r1_2a.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nWinner: {r1_2a.getWinner().name}")
print(f"Total rounds: {election_2a.getNumberOfIterations()}")

---
## Case 2b: Two Parties — Win in the Second Round

| Round | A  | B  | Result                          |
|:-----:|:--:|:--:|:--------------------------------:|
| 1     | 51 | 49 | no winner (51 % < 52 %)         |
| 2     | 55 | 45 | **A wins (55 % ≥ 52 %)**        |

In the first round A narrowly misses the threshold. In the second round it is enough.

In [ ]:
config_2b = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
)

election_2b = Election(electionConfig=config_2b)

r1_2b = election_2b.start(votes={'A': 51, 'B': 49})
print(f"Round 1: Winner={r1_2b.getWinner()}  → second round required")

r2_2b = election_2b.runNextIteration(
    r1_2b.getNextRoundInput().with_votes({'A': 55, 'B': 45})
)
print(f"Round 2: Winner={r2_2b.getWinner().name}")
print(f"Total rounds: {election_2b.getNumberOfIterations()}")

---
## Case 2c: Two Parties — Draw of Lots

Two parties where no winner can be determined in **two consecutive rounds** automatically trigger a draw of lots in the third round — as a reduction to fewer than two parties is not possible.

| Round | A  | B  | Result                      |
|:-----:|:--:|:--:|:---------------------------:|
| 1     | 51 | 49 | no winner (51 % < 52 %)     |
| 2     | 51 | 49 | no winner                   |
| 3     | —  | —  | **Draw of lots**            |

### Case 2cI: `DrawLotsStrategy.RANDOM`

The winner is determined by a uniformly distributed random draw. With a fixed `seed` the result is reproducible.

In [ ]:
config_2cI = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
    draw_lots_strategy=DrawLotsStrategy.RANDOM,
    seed=42,
)

election_2cI = Election(electionConfig=config_2cI)
votes = {'A': 51, 'B': 49}

r1_I = election_2cI.start(votes=votes)
r2_I = election_2cI.runNextIteration(r1_I.getNextRoundInput().with_votes(votes))

print(f"Round 2 → draw of lots in next round: {r2_I.needsDecisionByLotInNextRound()}")

r3_I = election_2cI.runNextIteration(r2_I.getNextRoundInput())

print(f"Round 3 ({type(r3_I).__name__}): Winner={r3_I.getWinner().name}")
print(f"Decided by lot: {r3_I.wasDecidedByLot()}")
print(f"Total rounds: {election_2cI.getNumberOfIterations()}")

### Case 2cII: `DrawLotsStrategy.MARGINAL_LEAD`

The marginal vote difference is treated as having arisen randomly. The party with the higher vote share in the preceding round wins — here deterministically A, since 51 > 49.

In [ ]:
config_2cII = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
)

election_2cII = Election(electionConfig=config_2cII)
votes = {'A': 51, 'B': 49}

r1_II = election_2cII.start(votes=votes)
r2_II = election_2cII.runNextIteration(r1_II.getNextRoundInput().with_votes(votes))
r3_II = election_2cII.runNextIteration(r2_II.getNextRoundInput())

print(f"Round 3 ({type(r3_II).__name__}): Winner={r3_II.getWinner().name}")
print(f"  → A wins deterministically (51 > 49), no random draw required")
print(f"Decided by lot: {r3_II.wasDecidedByLot()}")

---
## Case 3: Three Parties — Elimination Followed by Draw of Lots

This case combines two mechanisms: first a party is eliminated by the ⅔ rule, then two remaining parties lead to a draw of lots.

| Round | A  | B  | C  | Result                                         |
|:-----:|:--:|:--:|:--:|:----------------------------------------------:|
| 1     | 40 | 38 | 22 | no winner; C eliminated (⅔ rule)               |
| 2     | 51 | 49 | —  | no winner                                      |
| 3     | 51 | 49 | —  | no winner                                      |
| 4     | —  | —  | —  | **Draw of lots**                               |

**⅔ rule in round 1:** A (40 %) + B (38 %) = 78 % ≥ 66.7 % → C (22 %) is eliminated.  
A draw of lots is only triggered after **two** consecutive rounds with the same two parties without a winner.

In [ ]:
config_3 = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C'],
    draw_lots_strategy=DrawLotsStrategy.RANDOM,
    seed=7,
)

election_3 = Election(electionConfig=config_3)
ab_votes = {'A': 51, 'B': 49}

# Round 1: C is eliminated
r1_3 = election_3.start(votes={'A': 40, 'B': 38, 'C': 22})
print(f"Round 1: Still in the running after elimination: {list(r1_3.getNextRoundInput().contestants.keys())}")

# Rounds 2 and 3: A and B without a winner
r2_3 = election_3.runNextIteration(r1_3.getNextRoundInput().with_votes(ab_votes))
r3_3 = election_3.runNextIteration(r2_3.getNextRoundInput().with_votes(ab_votes))

print(f"Round 2: Winner={r2_3.getWinner()}")
print(f"Round 3: Winner={r3_3.getWinner()}  → draw of lots in next round: {r3_3.needsDecisionByLotInNextRound()}")

# Round 4: Draw of lots
r4_3 = election_3.runNextIteration(r3_3.getNextRoundInput())

print(f"\nRound 4 ({type(r4_3).__name__}): Winner={r4_3.getWinner().name}")
print(f"Decided by lot: {r4_3.wasDecidedByLot()}")
print(f"Total rounds: {election_3.getNumberOfIterations()}")

---
## Case 4: More Parties than Constituencies

When more parties participate than there are constituencies, not every party
with parliamentary seats can be assigned a constituency. Seat distribution
still follows proportional vote share — but constituency assignment applies
to at most as many parties as there are constituencies.

**Configuration:** 4 parties (A–D), 3 constituencies of 100 votes each →
6 parliamentary seats, ballot majority 52 %, parliamentary majority threshold 55 %.

Votes (round 1):

| Constituency | A                  | B                  | C                  | D                 |
|:------------:|:------------------:|:------------------:|:------------------:|:-----------------:|
| WK1          | 56                 | 24                 | 17                 | 3                 |
| WK2          | 56                 | 24                 | 17                 | 3                 |
| WK3          | 55                 | 25                 | 16                 | 4                 |
| **Total**    | **167 (55.7 %)** | **73 (24.3 %)** | **50 (16.7 %)** | **10 (3.3 %)** |

A wins directly in the first round (55.7 % ≥ 52 %). Since A's vote share
exceeds the parliamentary majority threshold (55.7 % > 55 %), all seats are
distributed proportionally.

Party D receives no seats due to insufficient votes. Party C receives one seat
but — since there are only 3 constituencies for 3 seat-eligible parties — no
constituency: C is represented in parliament without holding a direct mandate.

In [ ]:
from IPython.display import display

cc_f4 = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['WK1', 'WK2', 'WK3'],
    'constituency_size':  [100, 100, 100],
    'turnout_percent':    [100.0, 100.0, 100.0],
}))

config_f4 = ElectionConfig(
    constituencies_config=cc_f4,
    participating_parties=['A', 'B', 'C', 'D'],
    seed=42,
)
print(f"Constituencies: {cc_f4.getM()},  "
      f"Parties: {len(config_f4.participating_parties)},  "
      f"Parliamentary seats: {config_f4.parliamentarySeats}")

election_f4 = Election(electionConfig=config_f4)
final_f4 = election_f4.start(votes={
    'WK1': {'A': 56, 'B': 24, 'C': 17, 'D': 3},
    'WK2': {'A': 56, 'B': 24, 'C': 17, 'D': 3},
    'WK3': {'A': 55, 'B': 25, 'C': 16, 'D': 4},
})
print(f"Winner: {final_f4.getWinner().name}  "
      f"(Rounds: {election_f4.getNumberOfIterations()})")
display(final_f4.show_results_table(styler=True))

evaluator_f4 = ElectionEvaluator(seed=42)
result_f4 = evaluator_f4.evaluate(election_f4)

display(result_f4.get_seat_distribution_table())
display(result_f4.get_constituency_summary_table())
display(result_f4.get_constituency_assignment_table(sort_by='party'))